# Chapter 4 — Financial Feature Engineering: Alpha Factors

The book's chapter is about *how to research factors that predict returns*.
We do the same thing here, calibrated for crypto: build a comprehensive
factor library, compute it across the full eligible universe, and evaluate
each factor's predictive power, turnover, and standalone PnL.

What we build:

1. **A 19-factor library** covering momentum, mean-reversion, volatility,
   microstructure, drawdown, and cross-asset (β / relative strength to BTC).
2. **A factor panel** — every factor evaluated at every (symbol, ts) in our
   eligible universe.
3. **Cross-sectional sanity checks** — distributions, correlations,
   non-null coverage.
4. **Information Coefficient (IC) analysis** — daily cross-sectional Spearman
   rank correlation between each factor and forward returns at 1, 5, and
   20-day horizons.
5. **Turnover analysis** — how often does each factor's ranking change?
   (Critical for cost-aware strategies in Ch 5.)
6. **Decile (quintile) backtests** — naïve, cost-free long-short top vs
   bottom for each factor. Identifies factors worth carrying forward.

**Runtime note.** The full pipeline takes ~60–90 seconds on a ~70-symbol
universe over 4 years of daily data. Most of that is the IC computation
(repeated cross-sectional rank correlations) and the 19 decile backtests.

## Setup

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from ml4t_crypto import (
    eligible_universe, load_bars,
    compute_factor_panel, PER_SYMBOL_FACTORS,
    forward_returns, factor_ic, factor_ic_at_horizons,
    factor_turnover, decile_backtest, all_decile_backtests,
    perf_stats,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.options.display.float_format = "{:,.4f}".format

## 1. What is a factor?

A **factor** is a number computed from past information that we believe carries
some signal about future returns. The good ones are typically:

- **Cross-sectional** — they rank assets against each other at a point in time.
- **Stationary in distribution** — the rank meaning is roughly comparable across
  regimes (a top-decile-momentum name in 2021 looks like a top-decile-momentum
  name in 2024 in terms of where it sits in the cross-section).
- **Persistent enough to trade** — they don't flip sign every day.
- **Predictive at a useful horizon** — for crypto, that's typically 1–20 days.

Concretely, a factor is a function `f(bars[symbol, ≤t]) → x_{symbol,t}` whose
output we'll rank across symbols on each date. Long the top rank-bucket, short
the bottom rank-bucket → a long-short factor strategy.

## 2. Pull the universe and bars

We use the canonical filter from Ch 2: ≥1 year history, ≥\$1M trailing 20-day
median dollar volume, USDC quote (with USD fallback). For research we look back
to 2020-01-01 so we have meaningful sample for cross-asset windows up to 60
days, and we re-use the universe at the *start* of our research period.

In [ ]:
# Use a moderately liquid universe at the start of our research window.
# Setting the as-of date earlier ensures every name has plenty of in-sample
# data; setting it later would risk look-ahead via the universe filter.
AS_OF = "2024-01-01"
START = "2020-01-01"

eu = eligible_universe(AS_OF, min_history_days=365, min_dollar_volume=1_000_000)
syms = eu["symbol"].tolist()
print(f"{len(syms)} symbols pass the filter as of {AS_OF}.")
eu.head(10)

In [ ]:
# Load OHLCV bars for the entire universe over our research window.
bars = load_bars("1d", symbols=syms, start=START)
print(f"{len(bars):,} bars across {bars.index.get_level_values('symbol').nunique()} symbols, "
      f"{bars.index.get_level_values('ts').min().date()} → "
      f"{bars.index.get_level_values('ts').max().date()}")

## 3. The factor library

`ml4t_crypto.features` has 17 per-symbol factors plus 2 cross-asset factors
(β and relative strength to BTC). Categories:

| Category | Factors |
|---|---|
| Momentum / trend | `mom_20`, `mom_60`, `mom_120`, `mom_252`, `mom_12_1`, `trend_strength_60`, `dist_above_ma_50` |
| Reversal | `rev_1`, `rev_5` |
| Volatility | `vol_20`, `vol_60`, `vol_ratio_20_60`, `downside_vol_60` |
| Microstructure / volume | `dvol_zscore_20`, `amihud_20`, `vol_price_corr_20` |
| Drawdown | `drawdown_252` |
| Cross-asset (vs BTC-USDC) | `beta_to_ref_60`, `rel_strength_to_ref_60` |

`compute_factor_panel(bars)` runs all 19 over every `(symbol, ts)` in the
input and returns a long DataFrame.

In [ ]:
# Quick sanity check on what's in the library before we compute.
print(f"Per-symbol factors: {len(PER_SYMBOL_FACTORS)}")
for name, fn in PER_SYMBOL_FACTORS.items():
    print(f"  {name:25s} {fn.__doc__.splitlines()[0]}")

In [ ]:
# Compute the full panel — this is the expensive step (~10s for 68 symbols).
import time
t0 = time.time()
panel = compute_factor_panel(bars)
print(f"Computed in {time.time() - t0:.1f}s.")
print(f"Shape: {panel.shape}  (rows = (symbol × date), cols = factors)")
print(f"Columns: {list(panel.columns)}")
panel.head(3)

## 4. Sanity checks

Three quick things before we trust anything: coverage, distribution, and how
correlated the factors are with each other.

In [ ]:
# Coverage — non-null fraction per factor.
coverage = (panel.notna().mean() * 100).round(1).sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
coverage.plot.barh(ax=ax, color="C0")
ax.set_xlabel("% non-null")
ax.set_title("Per-factor coverage across the universe × time")
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

**Reading this.** Factors with longer lookbacks (`mom_252`, `mom_12_1`) have
the lowest coverage because they're undefined until the symbol has 252+ bars
of history. Short-lookback factors (`rev_1`, `dvol_zscore_20`) are near 100%.
Both reasonable.

In [ ]:
# Cross-sectional distribution of one factor on one date — should be reasonably
# well-spread, not pathologically clustered.
sample_date = pd.Timestamp("2024-06-01", tz="UTC")
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, fac in zip(axes.flat, ["mom_60", "vol_60", "drawdown_252", "rev_5", "amihud_20", "beta_to_ref_60"]):
    try:
        vals = panel[fac].xs(sample_date, level="ts").dropna()
        ax.hist(vals, bins=20, color="C2", edgecolor="white", alpha=0.85)
        ax.set_title(f"{fac} on {sample_date.date()} (n={len(vals)})")
    except KeyError:
        ax.set_title(f"{fac} — no data on {sample_date.date()}")
plt.tight_layout()
plt.show()

In [ ]:
# Inter-factor correlation: how much do these factors overlap?
# Use cross-sectional ranks (so we measure rank-correlation, robust to scale).
ranks = panel.groupby(level="ts").rank()
corr = ranks.corr(method="spearman")

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, ax=ax, cbar_kws={"label": "Spearman ρ"})
ax.set_title("Inter-factor rank correlations (cross-sectional, all dates)")
plt.tight_layout()
plt.show()

**Reading the correlation heatmap.**

- Momentum factors at different lookbacks (`mom_20`, `mom_60`, `mom_120`,
  `mom_252`) are strongly positively correlated with each other — winners tend
  to keep winning across timeframes. Picking *one* of them is usually enough.
- Volatility factors (`vol_20`, `vol_60`, `downside_vol_60`) are highly
  correlated; same logic.
- Drawdown is strongly *negatively* correlated with momentum — the names with
  the worst drawdowns are typically the worst momentum names. Same alpha,
  different lens.
- Reversal (`rev_1`, `rev_5`) is essentially uncorrelated with longer-horizon
  momentum. That's the classic short-vs-long momentum decomposition.
- `beta_to_ref_60` and `vol_60` correlate (high-beta names tend to be high-vol)
  but not perfectly — there's idiosyncratic beta.

For a real strategy we'd pick a *low-correlation subset* — e.g., one momentum,
one vol, one microstructure, one cross-asset — rather than 19 correlated takes
on the same idea.

## 5. Information Coefficient — do these factors predict returns?

The **Information Coefficient (IC)** is the daily cross-sectional Spearman
rank correlation between a factor's values and *forward* returns. We compute
it per day, then average.

- **Mean IC > 0**: high factor ranks predict high forward returns.
- **Mean IC < 0**: high factor ranks predict *low* forward returns (still useful — go short on top).
- **Mean IC ≈ 0**: no edge in the cross-section at this horizon.

In equities, |mean IC| around 0.02–0.05 is meaningful. Crypto tends to be
noisier — values can swing more day-to-day, but mean IC magnitudes are
similar.

In [ ]:
# IC at three forward horizons: 1, 5, 20 days.
import time
t0 = time.time()
ic_h = factor_ic_at_horizons(panel, bars, horizons=(1, 5, 20))
print(f"Computed IC at 3 horizons in {time.time() - t0:.1f}s.")
ic_h.columns = [f"IC @ {h}d" for h in ic_h.columns]
ic_h.sort_values("IC @ 5d", ascending=False).round(4)

In [ ]:
# Bar chart — easier to scan than the table.
ic_to_plot = ic_h.copy()
ic_to_plot = ic_to_plot.sort_values("IC @ 5d")
fig, ax = plt.subplots(figsize=(11, 7))
ic_to_plot.plot.barh(ax=ax, color=["C0", "C2", "C3"], width=0.8)
ax.axvline(0, color="black", lw=1)
ax.set_title("Mean Information Coefficient by factor and horizon")
ax.set_xlabel("Mean daily Spearman IC")
ax.legend(title="Horizon")
plt.tight_layout()
plt.show()

**Reading this.** We're looking for:

1. **Factors with consistent sign across horizons.** A factor where IC at 1d, 5d, and 20d all agree directionally is more trustworthy than one where the sign flips.
2. **Factors where IC magnitude grows with horizon.** Some factors only work at short horizons (e.g. short-term reversal); others compound nicely over longer windows (momentum).
3. **Negative-IC factors are useful too** — they're shortable. For volatility-family factors, the low-vol anomaly says low-vol names outperform high-vol, so a *negative* mean IC on `vol_60` is the right finding.

The cross-asset factors (`beta_to_ref_60`, `rel_strength_to_ref_60`) often
show interesting behavior — high-beta names tend to have higher absolute
return *and* more volatility, so IC sign can flip between bull/bear regimes.
We'll look at regime-conditioning in Ch 13.

In [ ]:
# IC time series for a few selected factors — is the predictive power stable
# over time, or concentrated in particular regimes?
t0 = time.time()
fwd5 = forward_returns(bars, horizon=5)
ic_ts = factor_ic(panel, fwd5)
print(f"Computed full IC time series in {time.time() - t0:.1f}s.")

selected = ["mom_60", "rev_5", "vol_60", "drawdown_252", "amihud_20"]
fig, ax = plt.subplots(figsize=(11, 5))
for f in selected:
    if f in ic_ts.columns:
        # Smooth the daily IC with a 30-day rolling mean for readability.
        ax.plot(ic_ts.index, ic_ts[f].rolling(30).mean(), label=f, lw=1.2)
ax.axhline(0, color="black", lw=1)
ax.set_title("30-day rolling mean of daily IC (5-day forward returns)")
ax.set_ylabel("Smoothed IC")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 6. Turnover — how often does the ranking change?

For each factor, the **bin turnover** is the average daily share of names
that change quantile bin. Higher turnover = more rebalancing = more trading
costs. In Ch 5 we'll formalize the cost trade-off; here we just measure.

Rough interpretation for daily-binned 10-quantiles:

- ~0.10 (= 1/10) → factor is essentially uncorrelated with itself day-over-day; suspicious unless the signal is genuinely a short-horizon thing like overnight reversal.
- ~0.50 → moderately persistent.
- < 0.20 → very persistent (low-turnover style, e.g. long-horizon momentum).

In [ ]:
t0 = time.time()
to = factor_turnover(panel, n_quantiles=10)
print(f"Computed turnover in {time.time() - t0:.1f}s.")
to.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
to.sort_values().plot.barh(ax=ax, color="C4")
ax.axvline(0.1, color="grey", ls=":", label="random (1/n_quantiles = 0.10)")
ax.axvline(0.5, color="grey", ls="--", label="moderate (0.50)")
ax.set_title("Decile turnover per factor — % of names changing bin per day")
ax.set_xlabel("Mean daily bin-change fraction")
ax.legend()
plt.tight_layout()
plt.show()

**Reading this.**

- `rev_1` and `dvol_zscore_20` have the highest turnover — they're high-frequency
  signals that change every day. Their cost profile will be expensive.
- Vol-family factors and long-horizon momentum have the lowest turnover —
  they're slow-moving. A weekly rebalance captures most of their signal.

For Ch 5 (strategy evaluation), the *combination* of IC magnitude and
turnover determines whether a factor survives transaction costs. A high-IC
high-turnover factor and a moderate-IC low-turnover factor may net to similar
post-cost performance.

## 7. Decile long-short backtests

A naïve but informative test: for each factor, every week, go long the top
decile and short the bottom decile, equally-weighted, no costs.

This is a **lower-bound on usefulness, an upper-bound on PnL.** No costs, no
position constraints, no risk management → numbers are optimistic. But if a
factor can't earn a positive Sharpe here, it almost certainly can't survive
real costs in Ch 5.

In [ ]:
# Vol-family, amihud, and drawdown are factors where "low values are good"
# (low-vol anomaly, illiquidity premium etc.). For those we invert the
# long-high direction — long the *bottom* decile.
inverted = {
    "vol_20":          False,
    "vol_60":          False,
    "downside_vol_60": False,
    "amihud_20":       False,
    "drawdown_252":    False,
}

import time
t0 = time.time()
bt = all_decile_backtests(panel, bars, n_quantiles=10, rebal_days=5,
                          long_high_overrides=inverted)
print(f"Ran 19 decile backtests in {time.time() - t0:.1f}s.")
bt.round(3)

In [ ]:
# Bar chart of Sharpe per factor — at a glance.
fig, ax = plt.subplots(figsize=(11, 7))
bt_sorted = bt["sharpe"].sort_values()
colors = ["C3" if v < 0 else "C2" for v in bt_sorted]
bt_sorted.plot.barh(ax=ax, color=colors)
ax.axvline(0, color="black", lw=1)
ax.set_title("Decile L/S Sharpe by factor (weekly rebal, no costs)")
ax.set_xlabel("Annualized Sharpe")
plt.tight_layout()
plt.show()

In [ ]:
# Equity curves for the top 3 factors by Sharpe.
top3 = bt.sort_values("sharpe", ascending=False).head(3).index.tolist()
print(f"Top 3 by Sharpe: {top3}")

fig, ax = plt.subplots(figsize=(11, 5))
for f in top3:
    res = decile_backtest(panel[f], bars, n_quantiles=10, rebal_days=5,
                          long_high=inverted.get(f, True))
    if not res["equity"].empty:
        eq = res["equity"]
        ax.plot(eq.index, eq.values,
                label=f"{f}  (Sharpe {res['sharpe']:.2f}, ret {res['ann_return']:.0%})",
                lw=1.4)
ax.axhline(1.0, color="grey", lw=0.7, ls="--")
ax.set_title("Equity curves — top 3 factors, weekly L/S decile, no costs")
ax.set_ylabel("Equity (start = 1.0)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

**Reading this carefully.**

These are **cost-free** results on a deliberately small (~70-symbol) universe.
Real-world deductions you should keep in mind:

- A 30 bps/side cost (which is roughly the right number for top-decile USDC
  pairs) at weekly rebal across ~14 names = ~60 bps per week = ~30 bps × 52
  = ~16% annual return drag. That eats most of the Sharpes in this chart.
- The factors that survive *after costs* are the ones with both meaningful
  IC and *low* turnover. That's the key Ch 5 calculation.
- We've ignored short borrow costs entirely. In crypto, shorting most of
  these names via spot is impossible — perp futures or margin-borrow is the
  mechanism, with its own funding cost.
- The cross-asset factors carry significant beta to BTC even net — a true
  market-neutral factor would need beta-hedging.

## 8. What's next

**Chapter 5 — Strategy Evaluation.** We take the survivors from this chapter,
apply a proper cost model, do purged & embargoed cross-validation (Ch 6
material that's necessary for Ch 5), and compute realistic post-cost Sharpes.

Things deferred from Ch 4 that you'll want eventually:

- **Multi-period IC**: how does IC behave over 30/60/90-day forward horizons?
  Useful for slower strategies.
- **Conditional IC**: does the factor work better in high-vol vs low-vol regimes,
  bull vs bear, high-correlation vs low-correlation environments? Ch 13.
- **Factor combinations**: ridge / Lasso / GBM on the panel rather than
  trading one factor at a time. Ch 7 + Ch 11.
- **Industry / sector controls**: in equities you'd neutralize against
  sector. The crypto equivalent (L1s, DeFi, memecoins, …) is fuzzy and a
  real research project on its own.